In [1]:
import numpy as np
from numba import njit
import matplotlib.pyplot as plt

## Dynamical Critical Exponent Simulation for Two-Species Totally Asymmetric Diffusion on a Ring

This notebook attempts to numerically solve for the dynamic critical exponent of a two species totally asymmetric diffusion model on a ring to replicate the work of [Kaufmann](https://sigma-journal.com/2010/039/sigma10-039.pdf). It has already been established that for   

Note that we define the random initial state as:

$$A\sim 1, \quad B\sim 2, \quad 0\sim 3$$

The paper describes the random configuration with equal densities.  

In [2]:
@njit
def random_initial_state(L):
    n = L // 3
    state = np.empty(L, dtype=np.int8)

    for i in range(n):
        state[i] = 0  

    for i in range(n, 2 * n):
        state[i] = 1 

    for i in range(2 * n, L):
        state[i] = 2 

    for i in range(L - 1, 0, -1):
        j = np.random.randint(0, i + 1)
        temp = state[i]
        state[i] = state[j]
        state[j] = temp

    return state

In [3]:
@njit
def one_sweep(state):
    """
    One Monte Carlo sweep = L random bond-update attempts.

    Each bond is chosen randomly. If the local pair is one of

        A B, A 0, B 0,

    then it is swapped.
    """
    L = len(state)

    for _ in range(L):
        j = np.random.randint(0, L)
        k = j + 1

        if k == L:
            k = 0

        if state[j] < state[k]:
            temp = state[j]
            state[j] = state[k]
            state[k] = temp

In [4]:
@njit
def fourier_mode_A(state, cos_q, sin_q):
    """
    Computes the first Fourier mode of the A-particle density:

        X(t) = sum_j 1_{site j has A} exp(2 pi i j / L).

    We return real and imaginary parts separately for numba speed.
    """
    re = 0.0
    im = 0.0
    L = len(state)

    for j in range(L):
        if state[j] == 0:
            re += cos_q[j]
            im += sin_q[j]

    return re, im

In [5]:
@njit
def simulate_fourier_time_series(L, n_samples, sample_every, burn_sweeps):
    """
    Simulates one long trajectory and records the first Fourier mode
    of the A-particle density.

    Parameters
    ----------
    L : int
        Ring size. Must be divisible by 3.
    n_samples : int
        Number of Fourier-mode samples to record.
    sample_every : int
        Number of sweeps between measurements.
    burn_sweeps : int
        Number of sweeps discarded before measurement.
    """
    state = random_initial_state(L)

    q = 2.0 * np.pi / L
    cos_q = np.empty(L)
    sin_q = np.empty(L)

    for j in range(L):
        cos_q[j] = np.cos(q * j)
        sin_q[j] = np.sin(q * j)

    # Burn-in
    for _ in range(burn_sweeps):
        one_sweep(state)

    X = np.empty(n_samples, dtype=np.complex128)

    for n in range(n_samples):
        re, im = fourier_mode_A(state, cos_q, sin_q)
        X[n] = re + 1j * im

        for _ in range(sample_every):
            one_sweep(state)

    return X

In [6]:
def autocorrelation_fft(x):
    """
    Fast autocorrelation using FFT.

    Returns normalized autocorrelation:

        C(t) = <X(s+t) X(s)^*> / <|X(s)|^2>.
    """
    x = np.asarray(x)
    x = x - np.mean(x)

    n = len(x)
    size = 1 << (2 * n - 1).bit_length()

    fx = np.fft.fft(x, size)
    ac = np.fft.ifft(fx * np.conjugate(fx))[:n]

    # Unbiased normalization by number of contributing pairs
    ac = ac / np.arange(n, 0, -1)

    # Normalize so C(0) = 1
    ac = ac / ac[0]

    return ac

In [7]:
def estimate_relaxation_time(
    L,
    n_samples=60000,
    sample_every=1,
    burn_factor=30
):
    """
    Estimate the relaxation time tau(L) from the decay of the
    first Fourier-mode autocorrelation.

    We fit the envelope roughly as

        |C(t)| ~ exp(-t / tau).

    The fitting window is chosen automatically.
    """
    burn_sweeps = int(burn_factor * L ** 1.5)

    X = simulate_fourier_time_series(
        L=L,
        n_samples=n_samples,
        sample_every=sample_every,
        burn_sweeps=burn_sweeps
    )

    C = autocorrelation_fft(X)

    t = np.arange(len(C)) * sample_every
    envelope = np.abs(C)

    # Light smoothing to reduce Monte Carlo noise
    window = 7
    kernel = np.ones(window) / window
    envelope_smooth = np.convolve(envelope, kernel, mode="same")

    # Find first time where the envelope has decayed significantly
    crossings = np.where((t > 0) & (envelope_smooth < 0.2))[0]

    if len(crossings) > 0:
        end = crossings[0]
    else:
        end = int(min(len(t), 4 * L ** 1.5))

    # Fit where the decay is neither too early nor too noisy
    mask = (
        (t > 0)
        & (np.arange(len(t)) < end)
        & (envelope_smooth < 0.85)
        & (envelope_smooth > 0.25)
    )

    if np.sum(mask) < 8:
        mask = (
            (t > 0)
            & (np.arange(len(t)) < end)
            & (envelope_smooth < 0.9)
            & (envelope_smooth > 0.15)
        )

    slope, intercept = np.polyfit(t[mask], np.log(envelope_smooth[mask]), 1)

    tau = -1.0 / slope

    return tau

In [8]:
"""
Estimate z by computing tau(L) for several ring sizes and fitting

    tau(L) ~ L^z.

Returns
-------
z : float
    Estimated dynamical critical exponent.
taus : array
    Estimated relaxation times.
"""
L_values = np.arange(30, 500, 30)
sample_every=1
n_samples=60000
taus = []

for L in L_values:
    print(f"Simulating L = {L}...")

    tau = estimate_relaxation_time(
        L=L,
        n_samples=n_samples,
        sample_every=sample_every,
        burn_factor=burn_factor,
    )

    taus.append(tau)
    print(f"  tau({L}) ≈ {tau:.4f}")

taus = np.array(taus)

logL = np.log(L_values)
logtau = np.log(taus)

z, intercept = np.polyfit(logL, logtau, 1)

print()
print("Estimated exponent:")
print(f"  z ≈ {z:.4f}")
print()
print("The theoretical KPZ value is:")
print("  z = 3/2 = 1.5")

plt.figure(figsize=(6, 4))
plt.plot(logL, logtau, "o", label="Monte Carlo data")
plt.plot(
    logL,
    intercept + z * logL,
    "--",
    label=fr"fit slope $z \approx {z:.3f}$"
)
plt.xlabel(r"$\log L$")
plt.ylabel(r"$\log \tau(L)$")
plt.title(r"Finite-size scaling: $\tau(L) \sim L^z$")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
